# v6 - Balanced + Embedding Cache (Derm7pt + HAM10000)

Evolucao do v5. Duas mudancas principais:

**1. Balanceamento por corte de NEV**
- Remove 2000 linhas NEV do train do HAM (seed=42) -> reduz dominancia da classe majoritaria.
- Class weights `inverse_sqrt` mantidos (undersample COMPLEMENTA, nao substitui).

**2. Cache de embeddings (encoder congelado)**
- O encoder MedSigLIP eh 100% congelado: o embedding de cada imagem eh IDENTICO em toda epoca.
- No v5 o encoder reprocessava tudo 8x (timeout ~17s/batch).
- Aqui pre-computamos os embeddings UMA vez e treinamos so a cabeca em cima deles.
- Isso libera folga enorme de tempo: EPOCHS=40 (teto), early stopping (patience=6) e ReduceLROnPlateau.
- `augment=False` no train (necessario: com augment o embedding cacheado ficaria invalido).

**Selecao do melhor checkpoint:** macro-F1 no Derm7pt val (nao no val combinado).

**Avaliacao final:** Derm7pt test e HAM test reportados SEPARADAMENTE (dois dominios).

In [ ]:
import os
import sys
import site
import importlib
import subprocess

REPO_URL = "https://github.com/RodrigoAraujo12/melanoma-tcc.git"
REPO_DIR = "/kaggle/working/melanoma-tcc"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "transformers", "accelerate", "huggingface_hub"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()
site.main()

print("Setup OK")

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from collections import Counter
from sklearn.metrics import f1_score, classification_report
from kaggle_secrets import UserSecretsClient

from melanoma_tcc.data.preprocessing import (
    Derm7ptUnifiedDataset, HAM10000Dataset, CombinedDermDataset,
    classification_collate_fn,
    GROUP_TO_LABEL, LABEL_TO_GROUP, METADATA_DIM_V5,
    HAM_DX_TO_GROUP, ham10000_train_val_split,
)
from melanoma_tcc.model.classifier import build_dermclassifier
from melanoma_tcc.model.losses import FocalLoss, compute_class_weights
from melanoma_tcc.utils.metrics import compute_metrics, plot_confusion_matrix

secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret('mellanoma_TCC')

DERM7PT_DIR = "/kaggle/input/datasets/rodrigoadesouza/derm7-pt-dataset/release_v0"
DERM_META = f"{DERM7PT_DIR}/meta/meta.csv"
DERM_IMAGES = f"{DERM7PT_DIR}/images"
DERM_TRAIN_IDX = f"{DERM7PT_DIR}/meta/train_indexes.csv"
DERM_VAL_IDX = f"{DERM7PT_DIR}/meta/valid_indexes.csv"
DERM_TEST_IDX = f"{DERM7PT_DIR}/meta/test_indexes.csv"

HAM_DIR = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
HAM_META = f"{HAM_DIR}/HAM10000_metadata.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Derm7pt meta existe: {os.path.exists(DERM_META)}")
print(f"HAM meta existe: {os.path.exists(HAM_META)}")
print(f"Metadata dim v5: {METADATA_DIM_V5}")

In [ ]:
model, processor = build_dermclassifier(
    hf_token=HF_TOKEN,
    num_classes=5,
    metadata_dim=METADATA_DIM_V5,
    freeze_vision=True,
)
model = model.to(device)
model.vision_encoder = model.vision_encoder.to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Derm7pt: schema unificado. augment=False em TODOS (cache de embedding exige imagem fixa).
derm_train = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                   indexes_csv=DERM_TRAIN_IDX, augment=False, seed=42)
derm_val = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                 indexes_csv=DERM_VAL_IDX, augment=False)
derm_test = Derm7ptUnifiedDataset(DERM_META, DERM_IMAGES, processor,
                                  indexes_csv=DERM_TEST_IDX, augment=False)

# HAM10000: filtra unknowns + split por lesion_id (evita leakage paciente)
ham_train_df, ham_val_df = ham10000_train_val_split(HAM_META, val_ratio=0.15, seed=42,
                                                    filter_unknown=True)

# ---- CORTE DE 2000 NEVUS DO TRAIN HAM ----
ham_train_df = ham_train_df.copy()
ham_train_df['group'] = (ham_train_df['dx'].str.strip().str.lower()
                         .map(HAM_DX_TO_GROUP).fillna('MISC'))

def _train_dist(derm_ds, ham_df):
    return Counter(pd.concat([derm_ds.df['group'], ham_df['group']]))

print("Distribuicao TRAIN ANTES do corte:")
print(_train_dist(derm_train, ham_train_df))

nev_idx = ham_train_df.index[ham_train_df['group'] == 'NEV']
n_cut = min(2000, len(nev_idx))
drop_idx = pd.Series(nev_idx).sample(n=n_cut, random_state=42)
ham_train_df = ham_train_df.drop(index=drop_idx).reset_index(drop=True)
print(f"\nCortou {n_cut} NEV do HAM train (de {len(nev_idx)} disponiveis).")

print("\nDistribuicao TRAIN DEPOIS do corte:")
print(_train_dist(derm_train, ham_train_df))
# ------------------------------------------

ham_train = HAM10000Dataset(ham_train_df, HAM_DIR, processor, augment=False, seed=42)
ham_val = HAM10000Dataset(ham_val_df, HAM_DIR, processor, augment=False)

train_dataset = CombinedDermDataset([derm_train, ham_train])

print(f"\nTrain combinado: {len(train_dataset)} = derm({len(derm_train)}) + ham({len(ham_train)})")
print(f"Derm val: {len(derm_val)} | HAM val: {len(ham_val)} | Derm test: {len(derm_test)}")

In [ ]:
BATCH_SIZE = 32
EPOCHS = 40          # teto de seguranca; early stopping decide o fim real
PATIENCE = 6         # epocas sem melhora de macro-F1 (derm_val) -> para
LR = 5e-4
FOCAL_GAMMA = 2.0
LABEL_SMOOTHING = 0.05

# Class weights a partir do train JA cortado (inverse_sqrt mantido do v5)
train_groups = pd.concat([derm_train.df['group'], ham_train.df['group']])
group_counts = Counter(train_groups)
class_counts = [group_counts.get(LABEL_TO_GROUP[i], 1) for i in range(5)]
alpha = compute_class_weights(class_counts, mode="inverse_sqrt")
print(f"Class counts: {[(LABEL_TO_GROUP[i], class_counts[i]) for i in range(5)]}")
print(f"Class weights (alpha, mode=inverse_sqrt): {alpha.tolist()}")

criterion = FocalLoss(alpha=alpha, gamma=FOCAL_GAMMA, label_smoothing=LABEL_SMOOTHING)
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

In [ ]:
# ===== PRE-COMPUTA EMBEDDINGS (unica vez que o encoder roda) =====
@torch.no_grad()
def precompute_embeddings(model, dataset, batch_size=32):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                        collate_fn=classification_collate_fn,
                        num_workers=4, pin_memory=True)
    embs, metas, labels = [], [], []
    for i, batch in enumerate(loader):
        pooled = model.encode_image(batch['pixel_values'].to(device))  # [B, hidden]
        embs.append(pooled.cpu())
        metas.append(batch['metadata'])
        labels.append(batch['labels'])
        if i % 20 == 0:
            print(f"  batch {i}/{len(loader)}")
    return TensorDataset(torch.cat(embs), torch.cat(metas), torch.cat(labels))

print("Cacheando train...")
train_cached = precompute_embeddings(model, train_dataset, BATCH_SIZE)
print("Cacheando derm_val...")
derm_val_cached = precompute_embeddings(model, derm_val, BATCH_SIZE)
print("Cacheando ham_val...")
ham_val_cached = precompute_embeddings(model, ham_val, BATCH_SIZE)
print("Cacheando derm_test...")
derm_test_cached = precompute_embeddings(model, derm_test, BATCH_SIZE)

emb_dim = train_cached.tensors[0].shape[1]
print(f"\nEmbedding dim: {emb_dim} | train N={len(train_cached)}")

train_loader = DataLoader(train_cached, batch_size=BATCH_SIZE, shuffle=True)
derm_val_loader = DataLoader(derm_val_cached, batch_size=BATCH_SIZE, shuffle=False)
ham_val_loader = DataLoader(ham_val_cached, batch_size=BATCH_SIZE, shuffle=False)
derm_test_loader = DataLoader(derm_test_cached, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# ===== Treino so da cabeca a partir dos embeddings cacheados =====
def head_forward(model, emb, md):
    v_feat = model.vision_proj(emb.to(device))
    m_feat = model.metadata_encoder(md.float().to(device))
    return model.classifier(torch.cat([v_feat, m_feat], dim=-1))

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for emb, md, lb in loader:
            lb = lb.to(device)
            logits = head_forward(model, emb, md)
            loss = criterion(logits, lb)
            total_loss += loss.item() * lb.size(0)
            all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
            all_labels.extend(lb.cpu().tolist())
    n = len(loader.dataset)
    avg_loss = total_loss / n
    acc = sum(int(p == l) for p, l in zip(all_preds, all_labels)) / n
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, macro_f1, all_preds, all_labels

best_val_f1 = 0.0
best_state = None
best_epoch = 0
epochs_no_improve = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    model.vision_encoder.eval()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0
    for emb, md, lb in train_loader:
        lb = lb.to(device)
        logits = head_forward(model, emb, md)
        loss = criterion(logits, lb)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        optimizer.step()
        train_loss_sum += loss.item() * lb.size(0)
        train_correct += (logits.argmax(dim=-1) == lb).sum().item()
        train_total += lb.size(0)
    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    # Selecao/monitoramento no DERM_VAL (nao no val combinado)
    val_loss, val_acc, val_f1, _, _ = evaluate(model, derm_val_loader, criterion)
    scheduler.step(val_f1)
    cur_lr = optimizer.param_groups[0]['lr']
    history.append((epoch, train_loss, train_acc, val_loss, val_acc, val_f1, cur_lr))
    print(f"Epoch {epoch:2d}/{EPOCHS} | train_loss={train_loss:.4f} acc={train_acc:.3f} | "
          f"val_loss={val_loss:.4f} acc={val_acc:.3f} macroF1={val_f1:.4f} | lr={cur_lr:.2e}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch = epoch
        best_state = {k: v.detach().cpu().clone()
                      for k, v in model.state_dict().items() if 'vision_encoder' not in k}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping na epoch {epoch} (sem melhora ha {PATIENCE} epocas).")
            break

print(f"\nBest macro-F1 (derm_val): {best_val_f1:.4f} na epoch {best_epoch}")

In [ ]:
if best_state is not None:
    current = model.state_dict()
    current.update(best_state)
    model.load_state_dict(current, strict=False)
    print("Best model carregado.")

os.makedirs("/kaggle/working/derm-classifier-v6", exist_ok=True)
torch.save(best_state, "/kaggle/working/derm-classifier-v6/classifier_head.pt")
print("Salvou /kaggle/working/derm-classifier-v6/classifier_head.pt")

import json
with open("/kaggle/working/derm-classifier-v6/history.json", "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
# ===== AVALIACAO FINAL EM DOIS DOMINIOS SEPARADOS =====
TARGET_NAMES = ["BCC", "NEV", "MEL", "SK", "MISC"]

print("=" * 60)
print("DOMINIO 1: Derm7pt TEST (comparacao direta com v3/v4/v5)")
print("=" * 60)
d_loss, d_acc, d_f1, d_preds, d_labels = evaluate(model, derm_test_loader, criterion)
print(f"accuracy = {d_acc:.4f} | macro-F1 = {d_f1:.4f} | loss = {d_loss:.4f}\n")
print(classification_report(d_labels, d_preds, target_names=TARGET_NAMES, digits=4, zero_division=0))
plot_confusion_matrix(d_labels, d_preds,
                      save_path='/kaggle/working/derm-classifier-v6-derm-cm.png',
                      target_names=TARGET_NAMES)

print("\n" + "=" * 60)
print("DOMINIO 2: HAM TEST (ham_val, nao usado na selecao)")
print("=" * 60)
h_loss, h_acc, h_f1, h_preds, h_labels = evaluate(model, ham_val_loader, criterion)
print(f"accuracy = {h_acc:.4f} | macro-F1 = {h_f1:.4f} | loss = {h_loss:.4f}\n")
print(classification_report(h_labels, h_preds, target_names=TARGET_NAMES, digits=4, zero_division=0))
plot_confusion_matrix(h_labels, h_preds,
                      save_path='/kaggle/working/derm-classifier-v6-ham-cm.png',
                      target_names=TARGET_NAMES)

In [ ]:
# Salva predicoes dos dois dominios separadamente
derm_pred_df = pd.DataFrame({
    'true_label': d_labels, 'pred_label': d_preds,
    'true_group': [LABEL_TO_GROUP[l] for l in d_labels],
    'pred_group': [LABEL_TO_GROUP[p] for p in d_preds],
})
derm_pred_df.to_csv('/kaggle/working/derm_classifier_v6_derm_predictions.csv', index=False)

ham_pred_df = pd.DataFrame({
    'true_label': h_labels, 'pred_label': h_preds,
    'true_group': [LABEL_TO_GROUP[l] for l in h_labels],
    'pred_group': [LABEL_TO_GROUP[p] for p in h_preds],
})
ham_pred_df.to_csv('/kaggle/working/derm_classifier_v6_ham_predictions.csv', index=False)

print(f"Derm7pt test: {len(derm_pred_df)} predicoes salvas.")
print(f"  pred dist: {Counter(derm_pred_df['pred_group'])}")
print(f"  true dist: {Counter(derm_pred_df['true_group'])}")
print(f"\nHAM test: {len(ham_pred_df)} predicoes salvas.")
print(f"  pred dist: {Counter(ham_pred_df['pred_group'])}")
print(f"  true dist: {Counter(ham_pred_df['true_group'])}")